# 面试题：点击日志有位置偏差时，怎样做 IPS/SNIPS 反事实排序？

排在第 1 位的文档更容易被看见，naive CTR 会把曝光位置当相关性。本 Notebook 手写 Position-Based Model 日志、propensity、IPS/SNIPS、clipping、有效样本量、反事实 policy value、加权 PyTorch ranker、bootstrap 和发布合同。

合成日志知道真实 relevance，便于验证估计偏差；真实系统必须通过随机化/干预估计 propensity，不能从点击标签自证。

In [ ]:
import copy,hashlib,json,math,random,warnings  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。
SEED74=7401  # 计算并保存当前步骤的中间状态。
RNG74=np.random.default_rng(SEED74); torch.manual_seed(SEED74); torch.set_num_threads(1)  # 计算并保存当前步骤的中间状态。
def canonical74(x): return json.dumps(x,sort_keys=True,separators=(",",":"))  # 定义本节可复用的核心函数。
def sha74(x): return hashlib.sha256(x).hexdigest()  # 定义本节可复用的核心函数。
assert torch.get_num_threads()==1  # 用受控断言验证关键不变量。

## 1. PBM：点击 = 被检查 × 相关

对每个 query 有 6 个候选和真实 relevance probability。logging policy 主要按一个有偏 feature 排序，但以小概率随机打乱，保证 positivity。位置检查概率 `e_r` 随 rank 下降；click 从 `Bernoulli(e_r * relevance)` 产生。

propensity 是“该文档在该日志策略下得到当前位置/被检查的概率”中的建模量。本教学用已知 examination probability，生产需随机 swap 或专门实验估计。

In [ ]:
EXAM74=np.array([1.,.72,.50,.34,.24,.18],np.float64)  # 计算并保存当前步骤的中间状态。
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Impression74:  # 定义承载本节状态与行为的数据结构。
    query:int; doc:int; position:int; feature:float; relevance:float; click:int; propensity:float  # 执行当前语句以推进本节示例。
def make_logs74(n_queries=900):  # 定义本节可复用的核心函数。
    rows=[]  # 计算并保存当前步骤的中间状态。
    for q in range(n_queries):  # 遍历输入元素以累积或检查结果。
        quality=RNG74.normal(size=6); feature=quality+RNG74.normal(scale=.8,size=6); relevance=1/(1+np.exp(-quality))  # 计算并保存当前步骤的中间状态。
        order=RNG74.permutation(6) if RNG74.random()<.15 else np.argsort(-feature)  # 计算并保存当前步骤的中间状态。
        for pos,doc in enumerate(order):  # 遍历输入元素以累积或检查结果。
            click=int(RNG74.random()<EXAM74[pos]*relevance[doc]); rows.append(Impression74(q,int(doc),pos,float(feature[doc]),float(relevance[doc]),click,float(EXAM74[pos])))  # 计算并保存当前步骤的中间状态。
    return rows  # 返回当前分支计算出的结果。
logs74=make_logs74()  # 计算并保存当前步骤的中间状态。
assert len(logs74)==5400 and all(0<r.propensity<=1 for r in logs74)  # 用受控断言验证关键不变量。
assert {r.position for r in logs74}==set(range(6)) and {r.doc for r in logs74 if r.query==0}==set(range(6))  # 用受控断言验证关键不变量。
assert all(r.click in (0,1) and 0<r.relevance<1 for r in logs74)  # 用受控断言验证关键不变量。

## 2. naive CTR 的位置偏差

按位置汇总点击率会明显下降，即使 relevance 分布没有同样幅度下降。`click/propensity` 的条件期望回到 relevance，但权重在尾部变大、方差上升。

先用受控真值比较 naive CTR、IPS relevance estimate 和真实平均 relevance，证明校正方向。

In [ ]:
ctr_by_pos74=np.array([np.mean([r.click for r in logs74 if r.position==p]) for p in range(6)])  # 计算并保存当前步骤的中间状态。
rel_by_pos74=np.array([np.mean([r.relevance for r in logs74 if r.position==p]) for p in range(6)])  # 计算并保存当前步骤的中间状态。
ips_rel_by_pos74=np.array([np.mean([r.click/r.propensity for r in logs74 if r.position==p]) for p in range(6)])  # 计算并保存当前步骤的中间状态。
assert ctr_by_pos74[0]>ctr_by_pos74[-1] and np.all(np.diff(EXAM74)<0)  # 用受控断言验证关键不变量。
assert np.mean(np.abs(ips_rel_by_pos74-rel_by_pos74))<np.mean(np.abs(ctr_by_pos74-rel_by_pos74))  # 用受控断言验证关键不变量。
assert np.all((ctr_by_pos74>=0)&(ctr_by_pos74<=1)) and np.isfinite(ips_rel_by_pos74).all()  # 用受控断言验证关键不变量。

## 3. IPS、SNIPS、clipping 与 ESS

对 click loss 使用 `w=1/propensity`。IPS 保持无偏但高方差；SNIPS 用权重和归一化，有限样本有偏但更稳；clipping 限制最大权重进一步做 bias-variance tradeoff。ESS=`(Σw)²/Σw²` 描述加权后相当于多少独立样本。

任何 propensity 为 0 都违反 positivity，不能靠加一个 epsilon 假装可评估。

In [ ]:
def weighted_mean74(values,propensities,mode="ips",clip=None):  # 定义本节可复用的核心函数。
    v=np.asarray(values,float); p=np.asarray(propensities,float)  # 计算并保存当前步骤的中间状态。
    if v.shape!=p.shape or np.any(p<=0) or np.any(p>1): raise ValueError("propensity_contract")  # 按当前条件选择后续控制路径。
    w=1/p; w=np.minimum(w,clip) if clip is not None else w  # 计算并保存当前步骤的中间状态。
    estimate=float(np.mean(w*v)) if mode=="ips" else float(np.sum(w*v)/np.sum(w)) if mode=="snips" else None  # 计算并保存当前步骤的中间状态。
    if estimate is None: raise ValueError("estimator_mode")  # 按当前条件选择后续控制路径。
    ess=float(w.sum()**2/(w@w)); return estimate,ess  # 计算并保存当前步骤的中间状态。
clicks74=np.array([r.click for r in logs74]); props74=np.array([r.propensity for r in logs74])  # 计算并保存当前步骤的中间状态。
ips_click74,ess_ips74=weighted_mean74(clicks74,props74,"ips"); snips_click74,ess_snips74=weighted_mean74(clicks74,props74,"snips"); clipped_click74,ess_clip74=weighted_mean74(clicks74,props74,"snips",3.)  # 计算并保存当前步骤的中间状态。
assert all(math.isfinite(x) for x in (ips_click74,snips_click74,clipped_click74,ess_ips74))  # 用受控断言验证关键不变量。
assert ess_clip74>=ess_snips74 and 0<ess_snips74<=len(logs74)  # 用受控断言验证关键不变量。
try: weighted_mean74([1],[0]); raise AssertionError("zero propensity accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="propensity_contract"  # 捕获预期异常并验证失败分支。

## 4. 目标策略的反事实 value

目标策略按 feature 排序时，可以利用每条已展示记录估计“若文档被检查”的 relevance；这里比较每个 query top-1 候选的真实 relevance、naive click 和 inverse-examination click。由于 logging 已把每个候选都展示在某个位置，support 完整。

若 logging 只展示 top-k，目标策略选到从未曝光文档就无法无偏评估，需要探索或模型假设。

In [ ]:
grouped74={q:[r for r in logs74 if r.query==q] for q in range(900)}  # 计算并保存当前步骤的中间状态。
assert all(sorted(r.position for r in rows)==list(range(6)) for rows in grouped74.values())  # 用受控断言验证关键不变量。
assert all(np.allclose([r.propensity for r in sorted(rows,key=lambda r:r.position)],EXAM74) for rows in grouped74.values())  # 用受控断言验证关键不变量。
target_rows74=[]  # 计算并保存当前步骤的中间状态。
for q,rows in grouped74.items():  # 遍历输入元素以累积或检查结果。
    chosen=max(rows,key=lambda r:(r.feature,-r.doc)); target_rows74.append(chosen)  # 计算并保存当前步骤的中间状态。
true_value74=float(np.mean([r.relevance for r in target_rows74])); naive_value74=float(np.mean([r.click for r in target_rows74])); ips_value74=float(np.mean([r.click/r.propensity for r in target_rows74]))  # 计算并保存当前步骤的中间状态。
assert abs(ips_value74-true_value74)<abs(naive_value74-true_value74)  # 用受控断言验证关键不变量。
assert 0<true_value74<1 and 0<=naive_value74<=1 and ips_value74>naive_value74  # 用受控断言验证关键不变量。
assert len({r.query for r in target_rows74})==900  # 用受控断言验证关键不变量。

## 5. 用 IPS 加权训练一个小 ranker

模型从单个 feature 预测 relevance logit。naive BCE 把未点击都当负例并受位置影响；IPS BCE 对 click 正负都按 examination inverse weighting，是教学近似。更严谨的 counterfactual LTR 会使用 policy propensity 和 pair/listwise objective。

train/test 按 query 切分，reference truth 只用于受控评估，不进入 loss。

In [ ]:
class ClickRanker74(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self): super().__init__(); self.linear=nn.Linear(1,1)  # 定义本节可复用的核心函数。
    def forward(self,x):  # 定义本节可复用的核心函数。
        if x.ndim!=2 or x.shape[1]!=1 or not torch.isfinite(x).all(): raise ValueError("ranker_input")  # 按当前条件选择后续控制路径。
        return self.linear(x).squeeze(-1)  # 返回当前分支计算出的结果。
train_logs74=[r for r in logs74 if r.query<700]; test_logs74=[r for r in logs74 if r.query>=700]  # 计算并保存当前步骤的中间状态。
x_train74=torch.tensor([[r.feature] for r in train_logs74],dtype=torch.float32); y_train74=torch.tensor([r.click for r in train_logs74],dtype=torch.float32); p_train74=torch.tensor([r.propensity for r in train_logs74])  # 计算并保存当前步骤的中间状态。
def train_ranker74(weighted):  # 定义本节可复用的核心函数。
    torch.manual_seed(7); model=ClickRanker74(); opt=torch.optim.Adam(model.parameters(),lr=.05)  # 计算并保存当前步骤的中间状态。
    for _ in range(140):  # 遍历输入元素以累积或检查结果。
        logits=model(x_train74); raw=F.binary_cross_entropy_with_logits(logits,y_train74,reduction="none"); loss=(raw*(1/p_train74).clamp(max=5)).mean() if weighted else raw.mean()  # 计算并保存当前步骤的中间状态。
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()  # 计算并保存当前步骤的中间状态。
    return model  # 返回当前分支计算出的结果。
naive_model74=train_ranker74(False); ips_model74=train_ranker74(True)  # 计算并保存当前步骤的中间状态。
assert all(torch.isfinite(p).all() for p in ips_model74.parameters()) and len(test_logs74)==1200  # 用受控断言验证关键不变量。
assert set(r.query for r in train_logs74).isdisjoint(r.query for r in test_logs74)  # 用受控断言验证关键不变量。

## 6. 用真实 relevance 做受控 nDCG oracle

在测试 query 内按模型 score 排序，使用真实 relevance 计算 nDCG@6。这个真值线上不可得，只用于验证 IPS 实现。还比较 top-1 relevance 与 feature baseline。

真实系统需要人工 relevance、随机化实验或在线指标；不能用同一有偏 click 再证明校正模型更好。

In [ ]:
def ndcg_real74(model):  # 定义本节可复用的核心函数。
    vals=[]; top=[]  # 计算并保存当前步骤的中间状态。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        for q in range(700,900):  # 遍历输入元素以累积或检查结果。
            rows=grouped74[q]; scores=model(torch.tensor([[r.feature] for r in rows],dtype=torch.float32)); order=torch.argsort(scores,descending=True); rel=torch.tensor([r.relevance for r in rows]); gains=(2**rel-1); disc=1/torch.log2(torch.arange(6,dtype=torch.float32)+2); vals.append(float((gains[order]*disc).sum()/(torch.sort(gains,descending=True).values*disc).sum())); top.append(rows[int(order[0])].relevance)  # 计算并保存当前步骤的中间状态。
    return float(np.mean(vals)),float(np.mean(top))  # 返回当前分支计算出的结果。
naive_ndcg74,naive_top74=ndcg_real74(naive_model74); ips_ndcg74,ips_top74=ndcg_real74(ips_model74)  # 计算并保存当前步骤的中间状态。
assert 0<=naive_ndcg74<=1 and 0<=ips_ndcg74<=1  # 用受控断言验证关键不变量。
assert ips_ndcg74>=naive_ndcg74-.01 and ips_ndcg74>.9  # 用受控断言验证关键不变量。
assert ips_top74>.55 and math.isfinite(ips_top74)  # 用受控断言验证关键不变量。

## 7. Query-level bootstrap 与权重诊断

置信区间必须以随机化单位 query/user 重采样，不能把一条 impression 当独立样本。这里 bootstrap 目标策略 IPS value，并报告尾部权重、ESS 和 clipping 敏感性。

若区间很宽，应增加探索/样本，而不是只报点估计。propensity 模型误设需要单独 sensitivity analysis。

In [ ]:
contribution74=np.array([r.click/r.propensity for r in target_rows74]); boot_rng74=np.random.default_rng(7410); boot74=[]  # 计算并保存当前步骤的中间状态。
for _ in range(400): boot74.append(float(contribution74[boot_rng74.integers(0,len(contribution74),len(contribution74))].mean()))  # 遍历输入元素以累积或检查结果。
ci74=np.quantile(boot74,[.025,.975]); weights74=1/props74  # 计算并保存当前步骤的中间状态。
assert ci74[0]<ips_value74<ci74[1] and ci74[1]-ci74[0]>0  # 用受控断言验证关键不变量。
assert np.quantile(weights74,.99)<=1/EXAM74[-1]+1e-9 and weights74.max()==1/EXAM74[-1]  # 用受控断言验证关键不变量。
assert np.std(boot74)>0 and np.isfinite(ci74).all()  # 用受控断言验证关键不变量。

## 8. 发布、监控与面试总结

manifest 绑定 logging policy、随机探索、propensity source、position schema、clip、query split、模型 state 和离线结果。线上监控 propensity 支持、最大权重、ESS、位置 CTR 和 policy drift；旧日志不能用新 propensity 无版本地重算。

面试回答顺序：偏差因果图 → 随机化估 propensity → positivity → IPS/SNIPS/clip → query-level 评估 → 在线实验。明确 IPS 不能修复未观测混杂。

In [ ]:
def state_digest74(model):  # 定义本节可复用的核心函数。
    h=hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for k,v in sorted(model.state_dict().items()): a=v.detach().numpy(); h.update(k.encode()); h.update(str(a.dtype).encode()); h.update(a.tobytes())  # 遍历输入元素以累积或检查结果。
    return h.hexdigest()  # 返回当前分支计算出的结果。
manifest74={"artifact_id":"ips-ranker-v1","pbm":{"exam":EXAM74.tolist(),"exploration":.15},"estimator":{"weight":"1/examination","clip":5.,"unit":"query"},"model":"linear-1d","state":state_digest74(ips_model74),"train_query":[0,700],"test_query":[700,900],"metrics":{"ndcg":ips_ndcg74,"ess":ess_ips74}}  # 计算并保存当前步骤的中间状态。
TRUST74=MappingProxyType({manifest74["artifact_id"]:sha74(canonical74(manifest74).encode())})  # 计算并保存当前步骤的中间状态。
def load_ips74(m,model):  # 定义本节可复用的核心函数。
    actual=copy.deepcopy(m); actual["state"]=state_digest74(model)  # 计算并保存当前步骤的中间状态。
    if TRUST74.get(actual.get("artifact_id"))!=sha74(canonical74(actual).encode()): raise RuntimeError("untrusted_ips_model")  # 按当前条件选择后续控制路径。
    model.eval(); return model  # 执行当前语句以推进本节示例。
loaded74=load_ips74(manifest74,ips_model74)  # 计算并保存当前步骤的中间状态。
assert loaded74.training is False and isinstance(TRUST74,MappingProxyType)  # 用受控断言验证关键不变量。
forged74=ClickRanker74()  # 计算并保存当前步骤的中间状态。
try: load_ips74(manifest74,forged74); raise AssertionError("forged IPS model accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="untrusted_ips_model"  # 捕获预期异常并验证失败分支。
print({"true":round(true_value74,3),"naive":round(naive_value74,3),"ips":round(ips_value74,3),"ips_ndcg":round(ips_ndcg74,3),"ess":round(ess_ips74)})  # 执行当前语句以推进本节示例。

## 9. 失败模式、复杂度与来源

估计与训练对日志线性，方差由最小 propensity 主导。常见错误：用 CTR 当 relevance、propensity 从同一 click 拟合无校验、出现零支持、impression 级 bootstrap、过度 clipping 不披露、logging/target policy 混淆和把 IPS 当因果万能药。

- Joachims et al., [Unbiased Learning-to-Rank with Biased Feedback](https://www.cs.cornell.edu/people/tj/publications/joachims_etal_17a.pdf), WSDM 2017。
- Swaminathan & Joachims, [Counterfactual Risk Minimization](https://proceedings.mlr.press/v37/swaminathan15.html), ICML 2015。
- Wang et al., [Position Bias Estimation for Unbiased LTR](https://arxiv.org/abs/1804.05938)。